# MELBA 0.1.0: business sales regression

This notebook demonstrates MELBA on a **synthetic** dataset. The generated values do not come from a company or a real market. MELBA uses a quadratic feature map with Huber loss and L2 regularization. The notebook reports metrics on a held-out split.

The notebook is designed for use from a MELBA checkout or as a standalone Colab notebook. When no checkout is present, the bundled source files below are written into the Colab runtime and imported directly. The data, training, and evaluation cells were run locally; a live Colab session and the optional plotting cell were not tested here.


In [ ]:
%pip install -q numpy pandas matplotlib


In [ ]:
from pathlib import Path, PurePosixPath
import sys
import tempfile

SOURCE_FILES = {'melba/__init__.py': '"""MELBA: a small business-data regression library."""\n'
                      '\n'
                      'from .model import MELBA\n'
                      '\n'
                      '__version__ = "0.1.0"\n'
                      '\n'
                      '__all__ = ["MELBA", "__version__"]\n',
 'melba/core/__init__.py': '"""Learning, optimization, and input validation '
                           'primitives."""\n'
                           '\n',
 'melba/core/learning.py': '"""Huber objective and derivative used by MELBA\'s '
                           'trainer."""\n'
                           '\n'
                           'from __future__ import annotations\n'
                           '\n'
                           'import numpy as np\n'
                           '\n'
                           '\n'
                           'def huber_loss(residuals: np.ndarray, delta: float) -> '
                           'np.ndarray:\n'
                           '    """Compute elementwise Huber loss for residuals and '
                           'threshold ``delta``."""\n'
                           '    absolute = np.abs(residuals)\n'
                           '    return np.where(\n'
                           '        absolute <= delta,\n'
                           '        0.5 * residuals**2,\n'
                           '        delta * (absolute - 0.5 * delta),\n'
                           '    )\n'
                           '\n'
                           '\n'
                           'def huber_gradient(residuals: np.ndarray, delta: float) -> '
                           'np.ndarray:\n'
                           '    """Compute the derivative of Huber loss with respect '
                           'to each residual."""\n'
                           '    return np.clip(residuals, -delta, delta)\n'
                           '\n',
 'melba/core/optimization.py': '"""Full-batch gradient descent for the regularized '
                               'Huber objective."""\n'
                               '\n'
                               'from __future__ import annotations\n'
                               '\n'
                               'import numpy as np\n'
                               '\n'
                               'from .learning import huber_gradient, huber_loss\n'
                               '\n'
                               '\n'
                               'def optimize_huber(\n'
                               '    design: np.ndarray,\n'
                               '    target: np.ndarray,\n'
                               '    *,\n'
                               '    learning_rate: float,\n'
                               '    epochs: int,\n'
                               '    tolerance: float,\n'
                               '    l2_penalty: float,\n'
                               '    huber_delta: float,\n'
                               '    random_state: int | None,\n'
                               ') -> tuple[np.ndarray, list[float], bool]:\n'
                               '    """Fit weights using full-batch gradient descent.\n'
                               '\n'
                               '    ``design`` excludes the intercept. The intercept '
                               'is prepended here and is\n'
                               '    left out of the L2 penalty. The Frobenius norm '
                               'provides a safe upper bound\n'
                               "    for the smooth part's gradient Lipschitz "
                               'constant.\n'
                               '    """\n'
                               '    sample_count, feature_count = design.shape\n'
                               '    augmented = '
                               'np.column_stack((np.ones(sample_count), design))\n'
                               '\n'
                               '    generator = np.random.default_rng(random_state)\n'
                               '    weights = np.zeros(feature_count + 1)\n'
                               '    weights[1:] = generator.normal(0.0, 0.01, '
                               'size=feature_count)\n'
                               '\n'
                               '    lipschitz_bound = float(np.sum(augmented * '
                               'augmented) / sample_count + l2_penalty)\n'
                               '    step_size = learning_rate / max(lipschitz_bound, '
                               'np.finfo(float).eps)\n'
                               '\n'
                               '    def objective(current: np.ndarray) -> float:\n'
                               '        residuals = augmented @ current - target\n'
                               '        data_loss = '
                               'float(np.mean(huber_loss(residuals, huber_delta)))\n'
                               '        penalty = 0.5 * l2_penalty * '
                               'float(np.sum(current[1:] ** 2))\n'
                               '        return data_loss + penalty\n'
                               '\n'
                               '    history = [objective(weights)]\n'
                               '    converged = False\n'
                               '    for _ in range(epochs):\n'
                               '        residuals = augmented @ weights - target\n'
                               '        gradient = augmented.T @ '
                               'huber_gradient(residuals, huber_delta) / sample_count\n'
                               '        gradient[1:] += l2_penalty * weights[1:]\n'
                               '        weights -= step_size * gradient\n'
                               '\n'
                               '        current_loss = objective(weights)\n'
                               '        history.append(current_loss)\n'
                               '        if abs(history[-2] - current_loss) <= '
                               'tolerance:\n'
                               '            converged = True\n'
                               '            break\n'
                               '\n'
                               '    return weights, history, converged\n'
                               '\n',
 'melba/core/validation.py': '"""Input checks shared by the model and evaluation '
                             'metrics."""\n'
                             '\n'
                             'from __future__ import annotations\n'
                             '\n'
                             'from numbers import Integral, Real\n'
                             '\n'
                             'import numpy as np\n'
                             '\n'
                             '\n'
                             'def _real_array(values: object, name: str) -> '
                             'np.ndarray:\n'
                             '    try:\n'
                             '        raw = np.asarray(values)\n'
                             '    except (TypeError, ValueError) as exc:\n'
                             '        raise ValueError(f"{name} must be a non-empty '
                             'numeric array.") from exc\n'
                             '\n'
                             '    if raw.dtype.kind not in "biuf":\n'
                             '        raise ValueError(f"{name} must contain real '
                             'numeric values.")\n'
                             '    try:\n'
                             '        array = np.asarray(raw, dtype=float)\n'
                             '    except (TypeError, ValueError, OverflowError) as '
                             'exc:\n'
                             '        raise ValueError(f"{name} must contain real '
                             'numeric values.") from exc\n'
                             '    return array\n'
                             '\n'
                             '\n'
                             'def as_features(values: object, name: str = "X") -> '
                             'np.ndarray:\n'
                             '    """Return features as a finite, two-dimensional '
                             'float array."""\n'
                             '    array = _real_array(values, name)\n'
                             '    if array.ndim == 1:\n'
                             '        array = array.reshape(-1, 1)\n'
                             '    elif array.ndim != 2:\n'
                             '        raise ValueError(f"{name} must be one- or '
                             'two-dimensional.")\n'
                             '    if array.shape[0] == 0 or array.shape[1] == 0:\n'
                             '        raise ValueError(f"{name} must contain at least '
                             'one sample and one feature.")\n'
                             '    if not np.isfinite(array).all():\n'
                             '        raise ValueError(f"{name} must not contain NaN '
                             'or infinity.")\n'
                             '    return array\n'
                             '\n'
                             '\n'
                             'def as_target(values: object, name: str = "y") -> '
                             'np.ndarray:\n'
                             '    """Return a finite, one-dimensional target '
                             'array."""\n'
                             '    array = _real_array(values, name)\n'
                             '    if array.ndim == 2 and array.shape[1] == 1:\n'
                             '        array = array[:, 0]\n'
                             '    elif array.ndim != 1:\n'
                             '        raise ValueError(f"{name} must be '
                             'one-dimensional or a single-column array.")\n'
                             '    if array.size == 0:\n'
                             '        raise ValueError(f"{name} must contain at least '
                             'one sample.")\n'
                             '    if not np.isfinite(array).all():\n'
                             '        raise ValueError(f"{name} must not contain NaN '
                             'or infinity.")\n'
                             '    return array\n'
                             '\n'
                             '\n'
                             'def validate_training_data(X: object, y: object) -> '
                             'tuple[np.ndarray, np.ndarray]:\n'
                             '    """Validate a feature matrix and its matching '
                             'target."""\n'
                             '    features = as_features(X)\n'
                             '    target = as_target(y)\n'
                             '    if features.shape[0] != target.size:\n'
                             '        raise ValueError(\n'
                             '            "X and y must contain the same number of '
                             'samples; "\n'
                             '            f"got {features.shape[0]} and '
                             '{target.size}."\n'
                             '        )\n'
                             '    return features, target\n'
                             '\n'
                             '\n'
                             'def validate_positive_real(value: object, name: str, *, '
                             'allow_zero: bool = False) -> float:\n'
                             '    """Validate and normalize a finite positive real '
                             'parameter."""\n'
                             '    if isinstance(value, bool) or not isinstance(value, '
                             'Real):\n'
                             '        raise ValueError(f"{name} must be a finite real '
                             'number.")\n'
                             '    normalized = float(value)\n'
                             '    minimum_ok = normalized >= 0 if allow_zero else '
                             'normalized > 0\n'
                             '    if not np.isfinite(normalized) or not minimum_ok:\n'
                             '        qualifier = "non-negative" if allow_zero else '
                             '"positive"\n'
                             '        raise ValueError(f"{name} must be finite and '
                             '{qualifier}.")\n'
                             '    return normalized\n'
                             '\n'
                             '\n'
                             'def validate_epochs(value: object) -> int:\n'
                             '    if isinstance(value, bool) or not isinstance(value, '
                             'Integral) or value <= 0:\n'
                             '        raise ValueError("epochs must be a positive '
                             'integer.")\n'
                             '    return int(value)\n'
                             '\n'
                             '\n'
                             'def validate_random_state(value: object) -> int | None:\n'
                             '    if value is None:\n'
                             '        return None\n'
                             '    if isinstance(value, bool) or not isinstance(value, '
                             'Integral):\n'
                             '        raise ValueError("random_state must be a '
                             'non-negative integer or None.")\n'
                             '    normalized = int(value)\n'
                             '    if normalized < 0:\n'
                             '        raise ValueError("random_state must be a '
                             'non-negative integer or None.")\n'
                             '    return normalized\n'
                             '\n',
 'melba/economics/__init__.py': '"""Economic functions, analytical marginal values, '
                                'and elasticity tools."""\n'
                                '\n'
                                'from .differential import (\n'
                                '    CriticalPoint,\n'
                                '    PolynomialFunction,\n'
                                '    ProfitOptimizationResult,\n'
                                '    marginal_cost,\n'
                                '    marginal_profit,\n'
                                '    marginal_revenue,\n'
                                '    optimize_profit,\n'
                                '    polynomial_derivative,\n'
                                ')\n'
                                'from .elasticity import Elasticity, ElasticityResult\n'
                                'from .linear import (\n'
                                '    CostFunction,\n'
                                '    DemandFunction,\n'
                                '    ProfitFunction,\n'
                                '    RevenueFunction,\n'
                                '    SupplyFunction,\n'
                                ')\n'
                                '\n'
                                '__all__ = [\n'
                                '    "CostFunction",\n'
                                '    "CriticalPoint",\n'
                                '    "DemandFunction",\n'
                                '    "Elasticity",\n'
                                '    "ElasticityResult",\n'
                                '    "PolynomialFunction",\n'
                                '    "ProfitFunction",\n'
                                '    "ProfitOptimizationResult",\n'
                                '    "RevenueFunction",\n'
                                '    "SupplyFunction",\n'
                                '    "marginal_cost",\n'
                                '    "marginal_profit",\n'
                                '    "marginal_revenue",\n'
                                '    "optimize_profit",\n'
                                '    "polynomial_derivative",\n'
                                ']\n',
 'melba/economics/_validation.py': '"""Numeric input checks shared by the economics '
                                   'modules."""\n'
                                   '\n'
                                   'from __future__ import annotations\n'
                                   '\n'
                                   'import numpy as np\n'
                                   '\n'
                                   '\n'
                                   'def as_finite_real(value: object, name: str, *, '
                                   'nonnegative: bool = False) -> float:\n'
                                   '    """Validate one finite real number and return '
                                   'it as a Python float."""\n'
                                   '    try:\n'
                                   '        raw = np.asarray(value)\n'
                                   '    except (TypeError, ValueError) as exc:\n'
                                   '        raise ValueError(f"{name} must be a finite '
                                   'real number.") from exc\n'
                                   '    if raw.ndim != 0 or raw.dtype.kind not in '
                                   '"iuf":\n'
                                   '        raise ValueError(f"{name} must be a finite '
                                   'real number.")\n'
                                   '    try:\n'
                                   '        result = float(raw)\n'
                                   '    except (TypeError, ValueError, OverflowError) '
                                   'as exc:\n'
                                   '        raise ValueError(f"{name} must be a finite '
                                   'real number.") from exc\n'
                                   '    if not np.isfinite(result):\n'
                                   '        raise ValueError(f"{name} must be a finite '
                                   'real number.")\n'
                                   '    if nonnegative and result < 0:\n'
                                   '        raise ValueError(f"{name} must be '
                                   'non-negative.")\n'
                                   '    return result\n'
                                   '\n'
                                   '\n'
                                   'def as_real_values(\n'
                                   '    value: object,\n'
                                   '    name: str,\n'
                                   '    *,\n'
                                   '    nonnegative: bool = False,\n'
                                   ') -> np.ndarray:\n'
                                   '    """Validate a non-empty array of finite real '
                                   'values and return float64."""\n'
                                   '    try:\n'
                                   '        raw = np.asarray(value)\n'
                                   '    except (TypeError, ValueError) as exc:\n'
                                   '        raise ValueError(f"{name} must be a '
                                   'non-empty numeric value or array.") from exc\n'
                                   '    if raw.dtype.kind not in "iuf":\n'
                                   '        raise ValueError(f"{name} must contain '
                                   'real numeric values.")\n'
                                   '    try:\n'
                                   '        values = np.asarray(raw, dtype=float)\n'
                                   '    except (TypeError, ValueError, OverflowError) '
                                   'as exc:\n'
                                   '        raise ValueError(f"{name} must contain '
                                   'real numeric values.") from exc\n'
                                   '    if values.size == 0:\n'
                                   '        raise ValueError(f"{name} must not be '
                                   'empty.")\n'
                                   '    if not np.isfinite(values).all():\n'
                                   '        raise ValueError(f"{name} must contain '
                                   'only finite values.")\n'
                                   '    if nonnegative and np.any(values < 0):\n'
                                   '        raise ValueError(f"{name} must contain '
                                   'only non-negative values.")\n'
                                   '    return values\n'
                                   '\n'
                                   '\n'
                                   'def scalar_or_array(values: np.ndarray) -> float | '
                                   'np.ndarray:\n'
                                   '    """Return zero-dimensional results as floats '
                                   'and preserve array shapes."""\n'
                                   '    if values.ndim == 0:\n'
                                   '        return float(values)\n'
                                   '    return values\n'
                                   '\n'
                                   '\n'
                                   'def finite_result(values: np.ndarray, name: str) '
                                   '-> np.ndarray:\n'
                                   '    """Raise a useful error if arithmetic produced '
                                   'an unrepresentable result."""\n'
                                   '    if not np.isfinite(values).all():\n'
                                   '        raise OverflowError(f"{name} calculation '
                                   'produced a non-finite result.")\n'
                                   '    return values\n',
 'melba/economics/differential.py': '"""Analytical derivatives and bounded profit '
                                    'optimization for polynomials."""\n'
                                    '\n'
                                    'from __future__ import annotations\n'
                                    '\n'
                                    'from dataclasses import dataclass\n'
                                    'from numbers import Integral\n'
                                    '\n'
                                    'import numpy as np\n'
                                    '\n'
                                    'from ._validation import as_real_values, '
                                    'finite_result, scalar_or_array\n'
                                    'from .linear import CostFunction, '
                                    'RevenueFunction\n'
                                    '\n'
                                    '\n'
                                    '@dataclass(frozen=True, init=False)\n'
                                    'class PolynomialFunction:\n'
                                    '    """A polynomial whose coefficients are '
                                    'ordered from constant upward.\n'
                                    '\n'
                                    '    For example, ``PolynomialFunction([100, 10, '
                                    '2])`` represents\n'
                                    '    ``100 + 10Q + 2Q**2``. Inputs and outputs '
                                    'must be finite real numbers.\n'
                                    '    """\n'
                                    '\n'
                                    '    coefficients: tuple[float, ...]\n'
                                    '\n'
                                    '    def __init__(self, coefficients: object) -> '
                                    'None:\n'
                                    '        values = as_real_values(coefficients, '
                                    '"coefficients")\n'
                                    '        if values.ndim != 1:\n'
                                    '            raise ValueError("coefficients must '
                                    'be a one-dimensional sequence.")\n'
                                    '        coefficients_tuple = tuple(float(value) '
                                    'for value in values)\n'
                                    '        while len(coefficients_tuple) > 1 and '
                                    'coefficients_tuple[-1] == 0.0:\n'
                                    '            coefficients_tuple = '
                                    'coefficients_tuple[:-1]\n'
                                    '        object.__setattr__(self, "coefficients", '
                                    'coefficients_tuple)\n'
                                    '\n'
                                    '    @property\n'
                                    '    def degree(self) -> int:\n'
                                    '        """Return the polynomial\'s degree (zero '
                                    'for a constant polynomial)."""\n'
                                    '        return len(self.coefficients) - 1\n'
                                    '\n'
                                    '    def calculate(self, value: object) -> float | '
                                    'np.ndarray:\n'
                                    '        """Evaluate the polynomial using NumPy\'s '
                                    'ascending-coefficient convention."""\n'
                                    '        values = as_real_values(value, "value")\n'
                                    '        with np.errstate(over="ignore", '
                                    'invalid="ignore"):\n'
                                    '            result = '
                                    'np.polynomial.polynomial.polyval(values, '
                                    'self.coefficients)\n'
                                    '        result_array = np.asarray(result, '
                                    'dtype=float)\n'
                                    '        return '
                                    'scalar_or_array(finite_result(result_array, '
                                    '"Polynomial"))\n'
                                    '\n'
                                    '    def derivative(self, order: int = 1) -> '
                                    '"PolynomialFunction":\n'
                                    '        """Return the analytical derivative '
                                    'polynomial of the requested order."""\n'
                                    '        if isinstance(order, bool) or not '
                                    'isinstance(order, Integral) or order < 0:\n'
                                    '            raise ValueError("order must be a '
                                    'non-negative integer.")\n'
                                    '        values = np.asarray(self.coefficients, '
                                    'dtype=float)\n'
                                    '        for _ in range(int(order)):\n'
                                    '            if values.size == 1:\n'
                                    '                values = np.zeros(1, '
                                    'dtype=float)\n'
                                    '            else:\n'
                                    '                values = values[1:] * '
                                    'np.arange(1, values.size, dtype=float)\n'
                                    '        return PolynomialFunction(values)\n'
                                    '\n'
                                    '    def derivative_at(self, value: object, order: '
                                    'int = 1) -> float | np.ndarray:\n'
                                    '        """Evaluate the analytical derivative at '
                                    'one or more values."""\n'
                                    '        return '
                                    'self.derivative(order).calculate(value)\n'
                                    '\n'
                                    '\n'
                                    '@dataclass(frozen=True)\n'
                                    'class CriticalPoint:\n'
                                    '    """A stationary quantity and its '
                                    'second-derivative classification."""\n'
                                    '\n'
                                    '    quantity: float\n'
                                    '    profit: float\n'
                                    '    second_derivative: float\n'
                                    '    classification: str\n'
                                    '\n'
                                    '\n'
                                    '@dataclass(frozen=True)\n'
                                    'class ProfitOptimizationResult:\n'
                                    '    """Global maximum of a polynomial profit '
                                    'function on finite bounds."""\n'
                                    '\n'
                                    '    bounds: tuple[float, float]\n'
                                    '    maximum_profit: float\n'
                                    '    optimal_quantities: tuple[float, ...]\n'
                                    '    critical_points: tuple[CriticalPoint, ...]\n'
                                    '    flat_objective: bool = False\n'
                                    '\n'
                                    '\n'
                                    'def _require_polynomial(function: object, name: '
                                    'str) -> PolynomialFunction:\n'
                                    '    if not isinstance(function, '
                                    'PolynomialFunction):\n'
                                    '        raise TypeError(f"{name} must be a '
                                    'PolynomialFunction.")\n'
                                    '    return function\n'
                                    '\n'
                                    '\n'
                                    'def polynomial_derivative(\n'
                                    '    function: PolynomialFunction,\n'
                                    '    order: int = 1,\n'
                                    ') -> PolynomialFunction:\n'
                                    '    """Return the analytical derivative of a '
                                    '``PolynomialFunction``."""\n'
                                    '    return _require_polynomial(function, '
                                    '"function").derivative(order)\n'
                                    '\n'
                                    '\n'
                                    'def marginal_cost(\n'
                                    '    cost_function: PolynomialFunction | '
                                    'CostFunction,\n'
                                    '    quantity: object,\n'
                                    ') -> float | np.ndarray:\n'
                                    '    """Evaluate marginal cost for a polynomial or '
                                    'linear total-cost function."""\n'
                                    '    derivative_at = getattr(cost_function, '
                                    '"derivative_at", None)\n'
                                    '    if not callable(derivative_at):\n'
                                    '        raise TypeError("cost_function must '
                                    'provide derivative_at(quantity).")\n'
                                    '    return derivative_at(quantity)\n'
                                    '\n'
                                    '\n'
                                    'def marginal_revenue(\n'
                                    '    revenue_function: PolynomialFunction | '
                                    'RevenueFunction,\n'
                                    '    quantity: object,\n'
                                    '    *,\n'
                                    '    price: object | None = None,\n'
                                    ') -> float | np.ndarray:\n'
                                    '    """Evaluate marginal revenue for a polynomial '
                                    'or fixed-price revenue function.\n'
                                    '\n'
                                    '    ``price`` is required for '
                                    '``RevenueFunction``, where the derivative holds\n'
                                    '    price constant. Polynomial total-revenue '
                                    'functions already express price\n'
                                    '    as part of the quantity relationship and need '
                                    'no separate price argument.\n'
                                    '    """\n'
                                    '    if isinstance(revenue_function, '
                                    'RevenueFunction):\n'
                                    '        if price is None:\n'
                                    '            raise ValueError("price is required '
                                    'for RevenueFunction marginal revenue.")\n'
                                    '        return '
                                    'revenue_function.derivative_at(quantity, '
                                    'price=price)\n'
                                    '    if price is not None:\n'
                                    '        raise ValueError("price must be omitted '
                                    'for a polynomial revenue function.")\n'
                                    '    revenue = '
                                    '_require_polynomial(revenue_function, '
                                    '"revenue_function")\n'
                                    '    return revenue.derivative_at(quantity)\n'
                                    '\n'
                                    '\n'
                                    'def marginal_profit(\n'
                                    '    profit_function: PolynomialFunction,\n'
                                    '    quantity: object,\n'
                                    ') -> float | np.ndarray:\n'
                                    '    """Evaluate marginal profit ``dProfit/dQ`` '
                                    'for a polynomial profit function."""\n'
                                    '    profit = _require_polynomial(profit_function, '
                                    '"profit_function")\n'
                                    '    return profit.derivative_at(quantity)\n'
                                    '\n'
                                    '\n'
                                    'def _real_roots_in_bounds(\n'
                                    '    derivative: PolynomialFunction,\n'
                                    '    lower: float,\n'
                                    '    upper: float,\n'
                                    ') -> list[float]:\n'
                                    '    coefficients = '
                                    'np.asarray(derivative.coefficients, dtype=float)\n'
                                    '    if coefficients.size == 1:\n'
                                    '        return []\n'
                                    '    try:\n'
                                    '        roots = '
                                    'np.polynomial.polynomial.polyroots(coefficients)\n'
                                    '    except (np.linalg.LinAlgError, ValueError) as '
                                    'exc:\n'
                                    '        raise ArithmeticError(\n'
                                    '            "Could not resolve derivative roots; '
                                    'rescale the polynomial or use narrower bounds."\n'
                                    '        ) from exc\n'
                                    '    bound_scale = max(1.0, abs(lower), '
                                    'abs(upper))\n'
                                    '    interval_tolerance = 64 * np.finfo(float).eps '
                                    '* bound_scale\n'
                                    '    real_roots: list[float] = []\n'
                                    '    for root in roots:\n'
                                    '        real = float(np.real(root))\n'
                                    '        imaginary = float(np.imag(root))\n'
                                    '        if abs(imaginary) > 1e-9 * max(1.0, '
                                    'abs(real)):\n'
                                    '            continue\n'
                                    '        if lower - interval_tolerance <= real <= '
                                    'upper + interval_tolerance:\n'
                                    '            real_roots.append(min(upper, '
                                    'max(lower, real)))\n'
                                    '    real_roots.sort()\n'
                                    '    distinct: list[float] = []\n'
                                    '    for root in real_roots:\n'
                                    '        root_tolerance = 1e-8 * max(1.0, '
                                    'abs(root), abs(distinct[-1]) if distinct else '
                                    '0.0)\n'
                                    '        if not distinct or abs(root - '
                                    'distinct[-1]) > root_tolerance:\n'
                                    '            distinct.append(root)\n'
                                    '    return distinct\n'
                                    '\n'
                                    '\n'
                                    'def _classify_stationary_point(function: '
                                    'PolynomialFunction, quantity: float) -> '
                                    'tuple[float, str]:\n'
                                    '    second = function.derivative_at(quantity, '
                                    'order=2)\n'
                                    '    second_value = float(second)\n'
                                    '    second_coefficients = '
                                    'np.abs(np.asarray(function.derivative(2).coefficients))\n'
                                    '    with np.errstate(over="ignore", '
                                    'invalid="ignore"):\n'
                                    '        scale = '
                                    'float(np.polynomial.polynomial.polyval(abs(quantity), '
                                    'second_coefficients))\n'
                                    '    if not np.isfinite(scale):\n'
                                    '        return second_value, "inconclusive"\n'
                                    '    tolerance = 1e-10 * max(1.0, scale)\n'
                                    '    if second_value < -tolerance:\n'
                                    '        return second_value, "local_maximum"\n'
                                    '    if second_value > tolerance:\n'
                                    '        return second_value, "local_minimum"\n'
                                    '    return second_value, "inconclusive"\n'
                                    '\n'
                                    '\n'
                                    'def optimize_profit(\n'
                                    '    profit_function: PolynomialFunction,\n'
                                    '    bounds: tuple[float, float],\n'
                                    ') -> ProfitOptimizationResult:\n'
                                    '    """Find a global profit maximum on a finite, '
                                    'closed quantity interval.\n'
                                    '\n'
                                    '    All real roots of the analytical first '
                                    'derivative in the interval are\n'
                                    '    compared with both boundaries. A stationary '
                                    'point is labeled a local\n'
                                    '    maximum or minimum only when the '
                                    'second-derivative test is decisive; a\n'
                                    '    zero or numerically inconclusive curvature is '
                                    'reported as inconclusive.\n'
                                    '    """\n'
                                    '    function = '
                                    '_require_polynomial(profit_function, '
                                    '"profit_function")\n'
                                    '    validated_bounds = as_real_values(bounds, '
                                    '"bounds", nonnegative=True)\n'
                                    '    if validated_bounds.ndim != 1 or '
                                    'validated_bounds.size != 2:\n'
                                    '        raise ValueError("bounds must be a pair '
                                    'of finite non-negative quantities.")\n'
                                    '    lower, upper = (float(value) for value in '
                                    'validated_bounds)\n'
                                    '    if lower > upper:\n'
                                    '        raise ValueError("bounds must be ordered '
                                    'as (lower_quantity, upper_quantity).")\n'
                                    '\n'
                                    '    if function.degree == 0:\n'
                                    '        optimal = (lower,) if lower == upper else '
                                    '(lower, upper)\n'
                                    '        return ProfitOptimizationResult(\n'
                                    '            bounds=(lower, upper),\n'
                                    '            '
                                    'maximum_profit=function.coefficients[0],\n'
                                    '            optimal_quantities=optimal,\n'
                                    '            critical_points=(),\n'
                                    '            flat_objective=True,\n'
                                    '        )\n'
                                    '\n'
                                    '    derivative = function.derivative()\n'
                                    '    roots = _real_roots_in_bounds(derivative, '
                                    'lower, upper)\n'
                                    '    critical_points: list[CriticalPoint] = []\n'
                                    '    for quantity in roots:\n'
                                    '        curvature, classification = '
                                    '_classify_stationary_point(function, quantity)\n'
                                    '        profit = '
                                    'float(function.calculate(quantity))\n'
                                    '        critical_points.append(\n'
                                    '            CriticalPoint(\n'
                                    '                quantity=quantity,\n'
                                    '                profit=profit,\n'
                                    '                second_derivative=curvature,\n'
                                    '                classification=classification,\n'
                                    '            )\n'
                                    '        )\n'
                                    '\n'
                                    '    candidates = [lower]\n'
                                    '    for quantity in [*roots, upper]:\n'
                                    '        if not any(\n'
                                    '            abs(quantity - existing)\n'
                                    '            <= 64 * np.finfo(float).eps * '
                                    'max(1.0, abs(quantity), abs(existing))\n'
                                    '            for existing in candidates\n'
                                    '        ):\n'
                                    '            candidates.append(quantity)\n'
                                    '    values = [float(function.calculate(quantity)) '
                                    'for quantity in candidates]\n'
                                    '    maximum = max(values)\n'
                                    '    comparison_tolerance = 1e-10 * max(1.0, '
                                    'abs(maximum))\n'
                                    '    optimal = tuple(\n'
                                    '        quantity\n'
                                    '        for quantity, value in zip(candidates, '
                                    'values)\n'
                                    '        if abs(value - maximum) <= '
                                    'comparison_tolerance\n'
                                    '    )\n'
                                    '    return ProfitOptimizationResult(\n'
                                    '        bounds=(lower, upper),\n'
                                    '        maximum_profit=maximum,\n'
                                    '        optimal_quantities=optimal,\n'
                                    '        critical_points=tuple(critical_points),\n'
                                    '        flat_objective=False,\n'
                                    '    )\n',
 'melba/economics/elasticity.py': '"""Price-elasticity calculations for differentiable '
                                  'demand functions."""\n'
                                  '\n'
                                  'from __future__ import annotations\n'
                                  '\n'
                                  'from dataclasses import dataclass\n'
                                  '\n'
                                  'import numpy as np\n'
                                  '\n'
                                  'from ._validation import as_real_values, '
                                  'finite_result, scalar_or_array\n'
                                  '\n'
                                  '\n'
                                  '@dataclass(frozen=True)\n'
                                  'class ElasticityResult:\n'
                                  '    """Point price elasticity and its '
                                  'magnitude-based category."""\n'
                                  '\n'
                                  '    price: float | np.ndarray\n'
                                  '    quantity: float | np.ndarray\n'
                                  '    derivative: float | np.ndarray\n'
                                  '    elasticity: float | np.ndarray\n'
                                  '    category: str | np.ndarray\n'
                                  '\n'
                                  '    @property\n'
                                  '    def interpretation(self) -> str | np.ndarray:\n'
                                  '        """Explain how quantity responds '
                                  'proportionally to price."""\n'
                                  '        explanations = {\n'
                                  '            "elastic": "Quantity demanded changes '
                                  'proportionally more than price.",\n'
                                  '            "inelastic": "Quantity demanded changes '
                                  'proportionally less than price.",\n'
                                  '            "unitary elastic": "Quantity demanded '
                                  'changes in the same proportion as price.",\n'
                                  '        }\n'
                                  '        if isinstance(self.category, str):\n'
                                  '            return explanations[self.category]\n'
                                  '        return np.select(\n'
                                  '            [self.category == "elastic", '
                                  'self.category == "unitary elastic"],\n'
                                  '            [explanations["elastic"], '
                                  'explanations["unitary elastic"]],\n'
                                  '            default=explanations["inelastic"],\n'
                                  '        )\n'
                                  '\n'
                                  '\n'
                                  'class Elasticity:\n'
                                  '    """Price-elasticity calculations for linear or '
                                  'polynomial demand functions."""\n'
                                  '\n'
                                  '    @staticmethod\n'
                                  '    def price_demand(demand_function: object, '
                                  'price: object) -> ElasticityResult:\n'
                                  '        """Calculate ``(dQ/dP) * (P/Q)`` and '
                                  'interpret its absolute magnitude.\n'
                                  '\n'
                                  '        ``demand_function`` must provide '
                                  '``calculate(price)`` and\n'
                                  '        ``derivative_at(price)`` methods. Quantity '
                                  'must be strictly positive at\n'
                                  '        every requested price. Categories use a '
                                  'relative and absolute\n'
                                  '        tolerance of ``1e-9`` around unit '
                                  'elasticity.\n'
                                  '        """\n'
                                  '        calculate = getattr(demand_function, '
                                  '"calculate", None)\n'
                                  '        derivative_at = getattr(demand_function, '
                                  '"derivative_at", None)\n'
                                  '        if not callable(calculate) or not '
                                  'callable(derivative_at):\n'
                                  '            raise TypeError(\n'
                                  '                "demand_function must provide '
                                  'calculate(price) and derivative_at(price)."\n'
                                  '            )\n'
                                  '\n'
                                  '        prices = as_real_values(price, "price", '
                                  'nonnegative=True)\n'
                                  '        quantities = '
                                  'as_real_values(calculate(prices), "demand '
                                  'quantity")\n'
                                  '        derivatives = '
                                  'as_real_values(derivative_at(prices), "demand '
                                  'derivative")\n'
                                  '        try:\n'
                                  '            prices, quantities, derivatives = '
                                  'np.broadcast_arrays(\n'
                                  '                prices, quantities, derivatives\n'
                                  '            )\n'
                                  '        except ValueError as exc:\n'
                                  '            raise ValueError(\n'
                                  '                "Demand quantity and derivative '
                                  'must be broadcast-compatible with price."\n'
                                  '            ) from exc\n'
                                  '        if np.any(quantities <= 0):\n'
                                  '            raise ValueError(\n'
                                  '                "Demand quantity must be greater '
                                  'than zero to calculate price elasticity."\n'
                                  '            )\n'
                                  '        with np.errstate(over="ignore", '
                                  'invalid="ignore", divide="ignore"):\n'
                                  '            elasticity = derivatives * (prices / '
                                  'quantities)\n'
                                  '        elasticity = finite_result(elasticity, '
                                  '"Price elasticity")\n'
                                  '        magnitude = np.abs(elasticity)\n'
                                  '        unitary = np.isclose(magnitude, 1.0, '
                                  'rtol=1e-9, atol=1e-9)\n'
                                  '        categories = np.where(\n'
                                  '            unitary,\n'
                                  '            "unitary elastic",\n'
                                  '            np.where(magnitude > 1, "elastic", '
                                  '"inelastic"),\n'
                                  '        )\n'
                                  '\n'
                                  '        if elasticity.ndim == 0:\n'
                                  '            category: str | np.ndarray = '
                                  'str(categories.item())\n'
                                  '        else:\n'
                                  '            category = categories\n'
                                  '        return ElasticityResult(\n'
                                  '            price=scalar_or_array(prices),\n'
                                  '            quantity=scalar_or_array(quantities),\n'
                                  '            '
                                  'derivative=scalar_or_array(derivatives),\n'
                                  '            '
                                  'elasticity=scalar_or_array(elasticity),\n'
                                  '            category=category,\n'
                                  '        )\n',
 'melba/economics/linear.py': '"""Linear economic functions for demand, supply, cost, '
                              'revenue, and profit."""\n'
                              '\n'
                              'from __future__ import annotations\n'
                              '\n'
                              'from dataclasses import dataclass\n'
                              '\n'
                              'import numpy as np\n'
                              '\n'
                              'from ._validation import as_finite_real, '
                              'as_real_values, finite_result, scalar_or_array\n'
                              '\n'
                              '\n'
                              '@dataclass(frozen=True)\n'
                              'class DemandFunction:\n'
                              '    """Linear demand ``Qd = a - bP`` for non-negative '
                              'prices.\n'
                              '\n'
                              '    ``a`` is the quantity intercept and ``b`` is the '
                              'non-negative price\n'
                              '    coefficient. The raw linear formula is returned '
                              'even beyond its choke\n'
                              '    price; negative quantities signal that the formula '
                              'is outside its useful\n'
                              '    economic range.\n'
                              '    """\n'
                              '\n'
                              '    a: float\n'
                              '    b: float\n'
                              '\n'
                              '    def __post_init__(self) -> None:\n'
                              '        object.__setattr__(self, "a", '
                              'as_finite_real(self.a, "a", nonnegative=True))\n'
                              '        object.__setattr__(self, "b", '
                              'as_finite_real(self.b, "b", nonnegative=True))\n'
                              '\n'
                              '    def calculate(self, price: object) -> float | '
                              'np.ndarray:\n'
                              '        """Calculate quantity demanded at one price or '
                              'an array of prices."""\n'
                              '        prices = as_real_values(price, "price", '
                              'nonnegative=True)\n'
                              '        with np.errstate(over="ignore", '
                              'invalid="ignore"):\n'
                              '            quantities = self.a - self.b * prices\n'
                              '        return '
                              'scalar_or_array(finite_result(quantities, "Demand"))\n'
                              '\n'
                              '    def derivative_at(self, price: object) -> float | '
                              'np.ndarray:\n'
                              '        """Return ``dQd/dP`` at each supplied price '
                              '(constant at ``-b``)."""\n'
                              '        prices = as_real_values(price, "price", '
                              'nonnegative=True)\n'
                              '        return scalar_or_array(np.full_like(prices, '
                              '-self.b, dtype=float))\n'
                              '\n'
                              '\n'
                              '@dataclass(frozen=True)\n'
                              'class SupplyFunction:\n'
                              '    """Linear supply ``Qs = c + dP`` for non-negative '
                              'prices."""\n'
                              '\n'
                              '    c: float\n'
                              '    d: float\n'
                              '\n'
                              '    def __post_init__(self) -> None:\n'
                              '        object.__setattr__(self, "c", '
                              'as_finite_real(self.c, "c", nonnegative=True))\n'
                              '        object.__setattr__(self, "d", '
                              'as_finite_real(self.d, "d", nonnegative=True))\n'
                              '\n'
                              '    def calculate(self, price: object) -> float | '
                              'np.ndarray:\n'
                              '        """Calculate quantity supplied at one price or '
                              'an array of prices."""\n'
                              '        prices = as_real_values(price, "price", '
                              'nonnegative=True)\n'
                              '        with np.errstate(over="ignore", '
                              'invalid="ignore"):\n'
                              '            quantities = self.c + self.d * prices\n'
                              '        return '
                              'scalar_or_array(finite_result(quantities, "Supply"))\n'
                              '\n'
                              '    def derivative_at(self, price: object) -> float | '
                              'np.ndarray:\n'
                              '        """Return ``dQs/dP`` at each supplied price '
                              '(constant at ``d``)."""\n'
                              '        prices = as_real_values(price, "price", '
                              'nonnegative=True)\n'
                              '        return scalar_or_array(np.full_like(prices, '
                              'self.d, dtype=float))\n'
                              '\n'
                              '\n'
                              '@dataclass(frozen=True)\n'
                              'class CostFunction:\n'
                              '    """Linear total cost ``TC(Q) = fixed_cost + '
                              'variable_cost * Q``.\n'
                              '\n'
                              '    ``variable_cost`` is the variable cost per unit of '
                              'output.\n'
                              '    """\n'
                              '\n'
                              '    fixed_cost: float\n'
                              '    variable_cost: float\n'
                              '\n'
                              '    def __post_init__(self) -> None:\n'
                              '        object.__setattr__(\n'
                              '            self,\n'
                              '            "fixed_cost",\n'
                              '            as_finite_real(self.fixed_cost, '
                              '"fixed_cost", nonnegative=True),\n'
                              '        )\n'
                              '        object.__setattr__(\n'
                              '            self,\n'
                              '            "variable_cost",\n'
                              '            as_finite_real(self.variable_cost, '
                              '"variable_cost", nonnegative=True),\n'
                              '        )\n'
                              '\n'
                              '    def calculate(self, quantity: object) -> float | '
                              'np.ndarray:\n'
                              '        """Calculate total cost for one quantity or an '
                              'array of quantities."""\n'
                              '        quantities = as_real_values(quantity, '
                              '"quantity", nonnegative=True)\n'
                              '        with np.errstate(over="ignore", '
                              'invalid="ignore"):\n'
                              '            costs = self.fixed_cost + '
                              'self.variable_cost * quantities\n'
                              '        return scalar_or_array(finite_result(costs, '
                              '"Cost"))\n'
                              '\n'
                              '    def derivative_at(self, quantity: object) -> float '
                              '| np.ndarray:\n'
                              '        """Return marginal cost ``dTC/dQ``, constant at '
                              '``variable_cost``."""\n'
                              '        quantities = as_real_values(quantity, '
                              '"quantity", nonnegative=True)\n'
                              '        return scalar_or_array(np.full_like(quantities, '
                              'self.variable_cost, dtype=float))\n'
                              '\n'
                              '\n'
                              'class RevenueFunction:\n'
                              '    """Total revenue function ``TR = price * '
                              'quantity``."""\n'
                              '\n'
                              '    def calculate(self, price: object, quantity: '
                              'object) -> float | np.ndarray:\n'
                              '        """Calculate revenue; scalar and array inputs '
                              'follow NumPy broadcasting."""\n'
                              '        prices = as_real_values(price, "price", '
                              'nonnegative=True)\n'
                              '        quantities = as_real_values(quantity, '
                              '"quantity", nonnegative=True)\n'
                              '        try:\n'
                              '            prices, quantities = '
                              'np.broadcast_arrays(prices, quantities)\n'
                              '        except ValueError as exc:\n'
                              '            raise ValueError(\n'
                              '                "price and quantity must have '
                              'broadcast-compatible shapes; "\n'
                              '                f"got {prices.shape} and '
                              '{quantities.shape}."\n'
                              '            ) from exc\n'
                              '        with np.errstate(over="ignore", '
                              'invalid="ignore"):\n'
                              '            revenue = prices * quantities\n'
                              '        return scalar_or_array(finite_result(revenue, '
                              '"Revenue"))\n'
                              '\n'
                              '    def derivative_at(self, quantity: object, *, price: '
                              'object) -> float | np.ndarray:\n'
                              '        """Return ``dTR/dQ`` while holding the supplied '
                              'price constant."""\n'
                              '        quantities = as_real_values(quantity, '
                              '"quantity", nonnegative=True)\n'
                              '        prices = as_real_values(price, "price", '
                              'nonnegative=True)\n'
                              '        try:\n'
                              '            prices, quantities = '
                              'np.broadcast_arrays(prices, quantities)\n'
                              '        except ValueError as exc:\n'
                              '            raise ValueError(\n'
                              '                "price and quantity must have '
                              'broadcast-compatible shapes; "\n'
                              '                f"got {prices.shape} and '
                              '{quantities.shape}."\n'
                              '            ) from exc\n'
                              '        return scalar_or_array(prices)\n'
                              '\n'
                              '\n'
                              'class ProfitFunction:\n'
                              '    """Profit function ``profit = total_revenue - '
                              'total_cost``."""\n'
                              '\n'
                              '    def calculate(self, total_revenue: object, '
                              'total_cost: object) -> float | np.ndarray:\n'
                              '        """Calculate profit or loss from non-negative '
                              'revenue and total cost."""\n'
                              '        revenue = as_real_values(total_revenue, '
                              '"total_revenue", nonnegative=True)\n'
                              '        costs = as_real_values(total_cost, '
                              '"total_cost", nonnegative=True)\n'
                              '        try:\n'
                              '            revenue, costs = '
                              'np.broadcast_arrays(revenue, costs)\n'
                              '        except ValueError as exc:\n'
                              '            raise ValueError(\n'
                              '                "total_revenue and total_cost must have '
                              'broadcast-compatible shapes; "\n'
                              '                f"got {revenue.shape} and '
                              '{costs.shape}."\n'
                              '            ) from exc\n'
                              '        with np.errstate(over="ignore", '
                              'invalid="ignore"):\n'
                              '            profit = revenue - costs\n'
                              '        return scalar_or_array(finite_result(profit, '
                              '"Profit"))\n',
 'melba/evaluation/__init__.py': '"""Regression metrics used by '
                                 ':class:`melba.MELBA`."""\n'
                                 '\n'
                                 'from .metrics import mean_absolute_error, '
                                 'mean_squared_error, r2_score, '
                                 'root_mean_squared_error\n'
                                 '\n'
                                 '__all__ = [\n'
                                 '    "mean_absolute_error",\n'
                                 '    "mean_squared_error",\n'
                                 '    "root_mean_squared_error",\n'
                                 '    "r2_score",\n'
                                 ']\n',
 'melba/evaluation/metrics.py': '"""Numerically stable regression metrics with '
                                'explicit constant-target handling."""\n'
                                '\n'
                                'from __future__ import annotations\n'
                                '\n'
                                'import math\n'
                                '\n'
                                'import numpy as np\n'
                                '\n'
                                'from ..core.validation import as_target\n'
                                '\n'
                                '\n'
                                'def _matching_targets(y_true: object, y_pred: object) '
                                '-> tuple[np.ndarray, np.ndarray]:\n'
                                '    actual = as_target(y_true, "y_true")\n'
                                '    predicted = as_target(y_pred, "y_pred")\n'
                                '    if actual.size != predicted.size:\n'
                                '        raise ValueError(\n'
                                '            "y_true and y_pred must contain the same '
                                'number of samples; "\n'
                                '            f"got {actual.size} and '
                                '{predicted.size}."\n'
                                '        )\n'
                                '    return actual, predicted\n'
                                '\n'
                                '\n'
                                'def _scaled_residuals(actual: np.ndarray, predicted: '
                                'np.ndarray) -> tuple[float, np.ndarray]:\n'
                                '    """Return residuals scaled to avoid overflow in '
                                'subtraction and squaring."""\n'
                                '    with np.errstate(over="ignore", '
                                'invalid="ignore"):\n'
                                '        residuals = actual - predicted\n'
                                '    if np.isfinite(residuals).all():\n'
                                '        scale = float(np.max(np.abs(residuals)))\n'
                                '        if scale == 0.0:\n'
                                '            return 0.0, np.zeros_like(actual)\n'
                                '        return scale, residuals / scale\n'
                                '\n'
                                '    scale = max(float(np.max(np.abs(actual))), '
                                'float(np.max(np.abs(predicted))))\n'
                                '    with np.errstate(over="ignore", invalid="ignore", '
                                'divide="ignore"):\n'
                                '        normalized_residuals = actual / scale - '
                                'predicted / scale\n'
                                '    return scale, normalized_residuals\n'
                                '\n'
                                '\n'
                                'def _checked_scaled_product(scale: float, factor: '
                                'float, metric: str) -> float:\n'
                                '    """Scale a finite nonnegative factor, raising if '
                                'the result overflows float64."""\n'
                                '    if scale == 0.0 or factor == 0.0:\n'
                                '        return 0.0\n'
                                '    maximum = float(np.finfo(float).max)\n'
                                '    if factor > 1.0 and scale > maximum / factor:\n'
                                '        raise OverflowError(f"{metric} is outside the '
                                'finite float64 range.")\n'
                                '    result = scale * factor\n'
                                '    if not math.isfinite(result):\n'
                                '        raise OverflowError(f"{metric} is outside the '
                                'finite float64 range.")\n'
                                '    return float(result)\n'
                                '\n'
                                '\n'
                                'def mean_absolute_error(y_true: object, y_pred: '
                                'object) -> float:\n'
                                '    """Return mean absolute error without overflowing '
                                'during subtraction."""\n'
                                '    actual, predicted = _matching_targets(y_true, '
                                'y_pred)\n'
                                '    scale, residuals = _scaled_residuals(actual, '
                                'predicted)\n'
                                '    normalized_error = '
                                'float(np.mean(np.abs(residuals)))\n'
                                '    return _checked_scaled_product(scale, '
                                'normalized_error, "MAE")\n'
                                '\n'
                                '\n'
                                'def _root_mean_squared_error(actual: np.ndarray, '
                                'predicted: np.ndarray) -> float:\n'
                                '    scale, residuals = _scaled_residuals(actual, '
                                'predicted)\n'
                                '    normalized_mse = float(np.mean(residuals * '
                                'residuals))\n'
                                '    return _checked_scaled_product(scale, '
                                'math.sqrt(normalized_mse), "RMSE")\n'
                                '\n'
                                '\n'
                                'def mean_squared_error(y_true: object, y_pred: '
                                'object) -> float:\n'
                                '    """Return MSE, raising if its final value cannot '
                                'fit in a float64."""\n'
                                '    actual, predicted = _matching_targets(y_true, '
                                'y_pred)\n'
                                '    scale, residuals = _scaled_residuals(actual, '
                                'predicted)\n'
                                '    normalized_mse = float(np.mean(residuals * '
                                'residuals))\n'
                                '    if scale == 0.0 or normalized_mse == 0.0:\n'
                                '        return 0.0\n'
                                '\n'
                                '    scale_mantissa, scale_exponent = '
                                'math.frexp(scale)\n'
                                '    mse_mantissa, mse_exponent = '
                                'math.frexp(normalized_mse)\n'
                                '    try:\n'
                                '        mse = math.ldexp(\n'
                                '            scale_mantissa * scale_mantissa * '
                                'mse_mantissa,\n'
                                '            2 * scale_exponent + mse_exponent,\n'
                                '        )\n'
                                '    except OverflowError as exc:\n'
                                '        raise OverflowError("MSE is outside the '
                                'finite float64 range.") from exc\n'
                                '    if not math.isfinite(mse):\n'
                                '        raise OverflowError("MSE is outside the '
                                'finite float64 range.")\n'
                                '    return float(mse)\n'
                                '\n'
                                '\n'
                                'def root_mean_squared_error(y_true: object, y_pred: '
                                'object) -> float:\n'
                                '    """Return RMSE directly, without first computing '
                                'a possibly overflowing MSE."""\n'
                                '    actual, predicted = _matching_targets(y_true, '
                                'y_pred)\n'
                                '    return _root_mean_squared_error(actual, '
                                'predicted)\n'
                                '\n'
                                '\n'
                                'def r2_score(y_true: object, y_pred: object) -> '
                                'float:\n'
                                '    """Return R-squared using scaled sums of '
                                'squares.\n'
                                '\n'
                                '    Raises ``OverflowError`` when a finite input pair '
                                'implies an R-squared below\n'
                                '    the range representable by a finite float64.\n'
                                '    """\n'
                                '    actual, predicted = _matching_targets(y_true, '
                                'y_pred)\n'
                                '    if np.all(actual == actual[0]):\n'
                                '        return 1.0 if np.array_equal(actual, '
                                'predicted) else 0.0\n'
                                '\n'
                                '    residual_scale, residuals = '
                                '_scaled_residuals(actual, predicted)\n'
                                '    residual_sum_scaled = float(np.sum(residuals * '
                                'residuals))\n'
                                '    if residual_sum_scaled == 0.0:\n'
                                '        return 1.0\n'
                                '\n'
                                '    with np.errstate(over="ignore", '
                                'invalid="ignore"):\n'
                                '        target_differences = actual - actual[0]\n'
                                '    if np.isfinite(target_differences).all():\n'
                                '        target_scale = '
                                'float(np.max(np.abs(target_differences)))\n'
                                '        scaled_target = target_differences / '
                                'target_scale\n'
                                '    else:\n'
                                '        target_scale = float(np.max(np.abs(actual)))\n'
                                '        scaled_target = actual / target_scale - '
                                'actual[0] / target_scale\n'
                                '    centered_target = scaled_target - '
                                'np.mean(scaled_target)\n'
                                '    total_sum_scaled = float(np.sum(centered_target * '
                                'centered_target))\n'
                                '    if total_sum_scaled == 0.0:\n'
                                '        raise ArithmeticError("Could not resolve '
                                'target variance at float64 precision.")\n'
                                '\n'
                                '    log_ratio = (\n'
                                '        2.0 * (math.log(residual_scale) - '
                                'math.log(target_scale))\n'
                                '        + math.log(residual_sum_scaled)\n'
                                '        - math.log(total_sum_scaled)\n'
                                '    )\n'
                                '    max_log = math.log(float(np.finfo(float).max))\n'
                                '    if log_ratio > max_log:\n'
                                '        raise OverflowError("R-squared is outside the '
                                'finite float64 range.")\n'
                                '    min_log = math.log(float(np.nextafter(0.0, '
                                '1.0)))\n'
                                '    ratio = 0.0 if log_ratio < min_log else '
                                'math.exp(log_ratio)\n'
                                '    score = 1.0 - ratio\n'
                                '    if not math.isfinite(score):\n'
                                '        raise OverflowError("R-squared is outside the '
                                'finite float64 range.")\n'
                                '    return float(score)\n',
 'melba/model.py': '"""Public MELBA business-data regression estimator."""\n'
                   '\n'
                   'from __future__ import annotations\n'
                   '\n'
                   'import numpy as np\n'
                   '\n'
                   'from .core.optimization import optimize_huber\n'
                   'from .core.validation import (\n'
                   '    as_features,\n'
                   '    as_target,\n'
                   '    validate_epochs,\n'
                   '    validate_positive_real,\n'
                   '    validate_random_state,\n'
                   '    validate_training_data,\n'
                   ')\n'
                   'from .evaluation.metrics import (\n'
                   '    mean_absolute_error,\n'
                   '    mean_squared_error,\n'
                   '    r2_score,\n'
                   '    root_mean_squared_error,\n'
                   ')\n'
                   'from .utils.preprocessing import BusinessFeatureMap\n'
                   '\n'
                   '\n'
                   'class MELBA:\n'
                   '    """Fit a regularized Huber regression model to business data.\n'
                   '\n'
                   '    By default, the model adds squared and pairwise feature terms '
                   'to capture\n'
                   '    simple curvature and interactions. This is a classical '
                   'quadratic feature\n'
                   '    regression formulation, not a claim of a new learning method.\n'
                   '    """\n'
                   '\n'
                   '    def __init__(\n'
                   '        self,\n'
                   '        learning_rate: float = 0.5,\n'
                   '        epochs: int = 1000,\n'
                   '        tolerance: float = 1e-8,\n'
                   '        random_state: int | None = 42,\n'
                   '        l2_penalty: float = 0.01,\n'
                   '        huber_delta: float = 1.345,\n'
                   '        quadratic_terms: bool = True,\n'
                   '    ) -> None:\n'
                   '        self.learning_rate = validate_positive_real(learning_rate, '
                   '"learning_rate")\n'
                   '        if self.learning_rate > 1.0:\n'
                   '            raise ValueError("learning_rate must be greater than 0 '
                   'and at most 1.")\n'
                   '        self.epochs = validate_epochs(epochs)\n'
                   '        self.tolerance = validate_positive_real(tolerance, '
                   '"tolerance", allow_zero=True)\n'
                   '        self.random_state = validate_random_state(random_state)\n'
                   '        self.l2_penalty = validate_positive_real(l2_penalty, '
                   '"l2_penalty", allow_zero=True)\n'
                   '        self.huber_delta = validate_positive_real(huber_delta, '
                   '"huber_delta")\n'
                   '        if not isinstance(quadratic_terms, (bool, np.bool_)):\n'
                   '            raise ValueError("quadratic_terms must be a '
                   'boolean.")\n'
                   '        self.quadratic_terms = bool(quadratic_terms)\n'
                   '\n'
                   '        self.is_fitted_ = False\n'
                   '        self.n_features_in_: int | None = None\n'
                   '        self.feature_map_: BusinessFeatureMap | None = None\n'
                   '        self.weights_: np.ndarray | None = None\n'
                   '        self.target_mean_: float | None = None\n'
                   '        self.target_scale_: float | None = None\n'
                   '        self.loss_history_: list[float] = []\n'
                   '        self.loss_: float | None = None\n'
                   '        self.n_iter_: int = 0\n'
                   '        self.converged_: bool = False\n'
                   '\n'
                   '    def fit(self, X: object, y: object) -> "MELBA":\n'
                   '        """Validate data and optimize model parameters, then '
                   'return ``self``."""\n'
                   '        features, target = validate_training_data(X, y)\n'
                   '        feature_map = BusinessFeatureMap(self.quadratic_terms)\n'
                   '        mapped = feature_map.fit_transform(features)\n'
                   '\n'
                   '        with np.errstate(over="ignore", invalid="ignore", '
                   'divide="ignore"):\n'
                   '            target_mean = float(np.mean(target))\n'
                   '            target_scale = float(np.std(target))\n'
                   '        if not np.isfinite(target_mean) or not '
                   'np.isfinite(target_scale):\n'
                   '            raise ValueError("y values are too large to normalize '
                   'safely.")\n'
                   '        if target_scale <= np.finfo(float).eps:\n'
                   '            target_scale = 1.0\n'
                   '        with np.errstate(over="ignore", invalid="ignore", '
                   'divide="ignore"):\n'
                   '            normalized_target = (target - target_mean) / '
                   'target_scale\n'
                   '        if not np.isfinite(normalized_target).all():\n'
                   '            raise ValueError("y values are too large to normalize '
                   'safely.")\n'
                   '\n'
                   '        weights, history, converged = optimize_huber(\n'
                   '            mapped,\n'
                   '            normalized_target,\n'
                   '            learning_rate=self.learning_rate,\n'
                   '            epochs=self.epochs,\n'
                   '            tolerance=self.tolerance,\n'
                   '            l2_penalty=self.l2_penalty,\n'
                   '            huber_delta=self.huber_delta,\n'
                   '            random_state=self.random_state,\n'
                   '        )\n'
                   '\n'
                   '        self.feature_map_ = feature_map\n'
                   '        self.weights_ = weights\n'
                   '        self.target_mean_ = target_mean\n'
                   '        self.target_scale_ = target_scale\n'
                   '        self.n_features_in_ = features.shape[1]\n'
                   '        self.loss_history_ = history\n'
                   '        self.loss_ = history[-1]\n'
                   '        self.n_iter_ = len(history) - 1\n'
                   '        self.converged_ = converged\n'
                   '        self.is_fitted_ = True\n'
                   '        return self\n'
                   '\n'
                   '    def predict(self, X: object) -> np.ndarray:\n'
                   '        """Predict target values for one or more feature rows."""\n'
                   '        if not self.is_fitted_ or self.feature_map_ is None or '
                   'self.weights_ is None:\n'
                   '            raise RuntimeError("MELBA must be fitted before '
                   'calling predict.")\n'
                   '        features = as_features(X)\n'
                   '        mapped = self.feature_map_.transform(features)\n'
                   '        design = np.column_stack((np.ones(features.shape[0]), '
                   'mapped))\n'
                   '        assert self.target_mean_ is not None and '
                   'self.target_scale_ is not None\n'
                   '        with np.errstate(over="ignore", invalid="ignore"):\n'
                   '            predictions = design @ self.weights_ * '
                   'self.target_scale_ + self.target_mean_\n'
                   '        if not np.isfinite(predictions).all():\n'
                   '            raise ValueError("Prediction produced nonfinite '
                   'values; check the input scale.")\n'
                   '        return predictions\n'
                   '\n'
                   '    def evaluate(self, X: object, y: object) -> dict[str, float]:\n'
                   '        """Return MAE, MSE, RMSE, and R-squared on the supplied '
                   'data."""\n'
                   '        actual = as_target(y)\n'
                   '        predictions = self.predict(X)\n'
                   '        if actual.size != predictions.size:\n'
                   '            raise ValueError(\n'
                   '                "X and y must contain the same number of samples; '
                   '"\n'
                   '                f"got {predictions.size} and {actual.size}."\n'
                   '            )\n'
                   '        return {\n'
                   '            "mae": mean_absolute_error(actual, predictions),\n'
                   '            "mse": mean_squared_error(actual, predictions),\n'
                   '            "rmse": root_mean_squared_error(actual, predictions),\n'
                   '            "r2": r2_score(actual, predictions),\n'
                   '        }\n'
                   '\n',
 'melba/utils/__init__.py': '"""Feature preparation utilities."""\n'
                            '\n'
                            'from .preprocessing import BusinessFeatureMap\n'
                            '\n'
                            '__all__ = ["BusinessFeatureMap"]\n',
 'melba/utils/preprocessing.py': '"""Training-fitted standardization and quadratic '
                                 'business feature mapping."""\n'
                                 '\n'
                                 'from __future__ import annotations\n'
                                 '\n'
                                 'import numpy as np\n'
                                 '\n'
                                 '\n'
                                 'class BusinessFeatureMap:\n'
                                 '    """Standardize inputs and optionally add squared '
                                 'and pairwise terms."""\n'
                                 '\n'
                                 '    def __init__(self, quadratic_terms: bool = True) '
                                 '-> None:\n'
                                 '        self.quadratic_terms = quadratic_terms\n'
                                 '        self.n_features_in_: int | None = None\n'
                                 '        self.input_mean_: np.ndarray | None = None\n'
                                 '        self.input_scale_: np.ndarray | None = None\n'
                                 '        self.term_mean_: np.ndarray | None = None\n'
                                 '        self.term_scale_: np.ndarray | None = None\n'
                                 '\n'
                                 '    def _expand(self, standardized: np.ndarray) -> '
                                 'np.ndarray:\n'
                                 '        if not self.quadratic_terms:\n'
                                 '            return standardized\n'
                                 '        count = standardized.shape[1]\n'
                                 '        quadratic = [\n'
                                 '            standardized[:, left] * standardized[:, '
                                 'right]\n'
                                 '            for left in range(count)\n'
                                 '            for right in range(left, count)\n'
                                 '        ]\n'
                                 '        return np.column_stack((standardized, '
                                 '*quadratic))\n'
                                 '\n'
                                 '    def fit_transform(self, features: np.ndarray) -> '
                                 'np.ndarray:\n'
                                 '        self.n_features_in_ = features.shape[1]\n'
                                 '        with np.errstate(over="ignore", '
                                 'invalid="ignore"):\n'
                                 '            self.input_mean_ = np.mean(features, '
                                 'axis=0)\n'
                                 '            input_scale = np.std(features, axis=0)\n'
                                 '        if not np.isfinite(self.input_mean_).all() '
                                 'or not np.isfinite(input_scale).all():\n'
                                 '            raise ValueError("X values are too large '
                                 'to scale safely.")\n'
                                 '        self.input_scale_ = np.where(input_scale > '
                                 'np.finfo(float).eps, input_scale, 1.0)\n'
                                 '\n'
                                 '        with np.errstate(over="ignore", '
                                 'invalid="ignore", divide="ignore"):\n'
                                 '            standardized = (features - '
                                 'self.input_mean_) / self.input_scale_\n'
                                 '            expanded = self._expand(standardized)\n'
                                 '            self.term_mean_ = np.mean(expanded, '
                                 'axis=0)\n'
                                 '            term_scale = np.std(expanded, axis=0)\n'
                                 '        if (\n'
                                 '            not np.isfinite(expanded).all()\n'
                                 '            or not '
                                 'np.isfinite(self.term_mean_).all()\n'
                                 '            or not np.isfinite(term_scale).all()\n'
                                 '        ):\n'
                                 '            raise ValueError("X values are too large '
                                 'to scale safely.")\n'
                                 '        self.term_scale_ = np.where(term_scale > '
                                 'np.finfo(float).eps, term_scale, 1.0)\n'
                                 '        with np.errstate(over="ignore", '
                                 'invalid="ignore", divide="ignore"):\n'
                                 '            mapped = (expanded - self.term_mean_) / '
                                 'self.term_scale_\n'
                                 '        if not np.isfinite(mapped).all():\n'
                                 '            raise ValueError("X values are too large '
                                 'to scale safely.")\n'
                                 '        return mapped\n'
                                 '\n'
                                 '    def transform(self, features: np.ndarray) -> '
                                 'np.ndarray:\n'
                                 '        if self.n_features_in_ is None or '
                                 'self.input_mean_ is None:\n'
                                 '            raise RuntimeError("The feature map must '
                                 'be fitted before transform.")\n'
                                 '        if features.shape[1] != '
                                 'self.n_features_in_:\n'
                                 '            raise ValueError(\n'
                                 '                "X has a different number of '
                                 'features than the data used for training; "\n'
                                 '                f"expected {self.n_features_in_}, '
                                 'got {features.shape[1]}."\n'
                                 '            )\n'
                                 '        assert self.input_scale_ is not None\n'
                                 '        assert self.term_mean_ is not None\n'
                                 '        assert self.term_scale_ is not None\n'
                                 '        with np.errstate(over="ignore", '
                                 'invalid="ignore", divide="ignore"):\n'
                                 '            standardized = (features - '
                                 'self.input_mean_) / self.input_scale_\n'
                                 '            expanded = self._expand(standardized)\n'
                                 '            mapped = (expanded - self.term_mean_) / '
                                 'self.term_scale_\n'
                                 '        if not np.isfinite(mapped).all():\n'
                                 '            raise ValueError("X values are too large '
                                 'to transform safely.")\n'
                                 '        return mapped\n'
                                 '\n'}

project_root = Path.cwd()
if (project_root / "melba" / "__init__.py").is_file():
    source_root = project_root
else:
    source_root = Path(tempfile.mkdtemp(prefix="melba_source_"))
    for relative_path, source in SOURCE_FILES.items():
        relative = PurePosixPath(relative_path)
        if relative.is_absolute() or ".." in relative.parts or not relative.parts:
            raise ValueError(f"Invalid bundled source path: {relative_path!r}")
        destination = source_root.joinpath(*relative.parts)
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_text(source, encoding="utf-8")
sys.path.insert(0, str(source_root))
from melba import MELBA, __version__
print(f"MELBA {__version__} loaded from {source_root}")


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(2026)
sample_count = 320
price = rng.uniform(12.0, 60.0, sample_count)
promotion_spend = rng.uniform(0.0, 80.0, sample_count)
operating_cost = rng.uniform(500.0, 2500.0, sample_count)
sales = (
    280.0 - 2.0 * price + 1.3 * promotion_spend
    - 0.009 * promotion_spend**2 - 0.018 * operating_cost
    + 0.008 * price * promotion_spend
    + rng.normal(0.0, 8.0, sample_count)
)
data = pd.DataFrame({
    'price': price,
    'promotion_spend': promotion_spend,
    'operating_cost': operating_cost,
    'sales': sales,
})
data.head()


## Split training and test rows

The shuffled test rows stay out of `fit`. The same random seed makes this demonstration repeatable.


In [ ]:
order = rng.permutation(len(data))
split = int(0.8 * len(order))
train = data.iloc[order[:split]]
test = data.iloc[order[split:]]
feature_columns = ['price', 'promotion_spend', 'operating_cost']
X_train = train[feature_columns]
y_train = train['sales']
X_test = test[feature_columns]
y_test = test['sales']
print(f'train rows: {len(train)}, test rows: {len(test)}')


## Fit, predict, and evaluate


In [ ]:
model = MELBA(learning_rate=0.5, epochs=1800, random_state=2026)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
results = model.evaluate(X_test, y_test)
print('metrics:', results)
print('first five predictions:', predictions[:5])
print(f'optimizer updates: {model.n_iter_}, final objective: {model.loss_:.6f}')


In [ ]:
import matplotlib.pyplot as plt

lower = min(float(y_test.min()), float(predictions.min()))
upper = max(float(y_test.max()), float(predictions.max()))
plt.figure(figsize=(6, 5))
plt.scatter(y_test, predictions, alpha=0.75, color='#226c63')
plt.plot([lower, upper], [lower, upper], color='#a34f32', linewidth=1.5)
plt.xlabel('Actual synthetic sales')
plt.ylabel('Predicted sales')
plt.title('MELBA predictions on held-out synthetic data')
plt.tight_layout()
plt.show()


## Reading the result

Use the printed test metrics and scatter plot to inspect how closely predictions follow the synthetic targets. The diagonal line marks exact predictions. These results describe only this generated dataset; they do not establish performance on real business data or a causal effect of price, promotion, or cost.
